Die benötigten Bibliotheken werden importiert

In [ ]:
import random
import simpy

Definierte Konstanten bzw. Variablen

In [ ]:
RANDOM_SEED = 42
TANKSTELLEN_GROEßE = 200            # Größe des Tankstellentanks (Liter)
SCHWELLENWERT = 25                  # minimales Level an der Tankstelle (% von max) 
AUTO_TANK_GROEßE = 50               # Größe des Autotanks (Liter)
AUTO_TANK_LEVEL = [5, 25]           # Min/max Level des Autotanks (Liter)
AUFFUELLGESCHWINDIGKEIT = 2         # Auffüllgeschwindigkeit des Kraftstoffes beim Autotank (Liter/ Sekunde)
TANKWAGEN_ZEIT = 300                # Zeit die eine Tankwagen zum ankommen benötigt (Sekunden)
ANKUFNTSINTERVALL_AUTO = [30, 300]  # Intervall zwischen den Ankuftszeiten der Autos [min, max] (Sekunden)
SIMULATIONS_ZEIT = 1000             # Simulationszeit (Sekunden)

Der Generator auto() wird definiert. 

In [ ]:
def auto(name: str, env: simpy.Environment, tankstelle: simpy.Resource, tankstellen_tank: simpy.Container):
    """
    Simuliert den Tankvorgang eines einzelnen Autos an der Tankstelle.

    Parameter:
        name (str):
            Der Name oder die Bezeichnung des Autos (z. B. "Auto 1").
        env (simpy.Environment):
            Die Simulationsumgebung, welche die Zeit und Ereignisse steuert.
        tankstelle (simpy.Resource):
            Die Ressource, die die verfügbaren Zapfsäulen repräsentiert.
            Ein Auto muss eine Zapfsäule anfordern, bevor es tanken kann.
        tankstellen_tank (simpy.Container):
            Der gemeinsame Vorratstank der Tankstelle, aus dem der Kraftstoff entnommen wird.

    Funktionsweise:
        1. Das Auto trifft mit einem zufälligen Tankfüllstand an der Tankstelle ein.
        2. Es fordert eine freie Zapfsäule an (`tankstelle.request()`).
           - Wenn keine frei ist, muss das Auto warten.
        3. Sobald eine Zapfsäule frei ist:
           - Es wird berechnet, wie viel Kraftstoff benötigt wird, um den Tank zu füllen.
           - Diese Menge wird aus dem Tankstellentank entnommen (`tankstellen_tank.get()`).
           - Der Tankvorgang dauert eine bestimmte Zeit, abhängig von der Füllgeschwindigkeit.
        4. Nach Abschluss des Tankvorgangs wird eine Meldung ausgegeben,
           die den Zeitpunkt und die getankte Menge anzeigt.

    Zweck:
        Diese Funktion modelliert das Verhalten eines Autos, das an der Tankstelle ankommt,
        wartet, tankt und dann wieder abfährt. Sie ist ein zentraler Bestandteil der Simulation
        und zeigt das Zusammenspiel zwischen Fahrzeugen, Ressourcen (Zapfsäulen) und Vorräten (Tank).
    """
    
    auto_tank_level = random.randint(*AUTO_TANK_LEVEL)
    print(f'{env.now:6.2f} s: {name} kommt an der Tankstelle an')  # 6.2f formatierung von env.now 

    with tankstelle.request() as req:
        # Request eine Zapfsäule
        yield req

        # Bekommt die benötigte Menge an Kraftstoff
        benoetigter_kraftstoff = AUTO_TANK_GROEßE - auto_tank_level
        yield tankstellen_tank.get(benoetigter_kraftstoff)

        # Der Tankprozess benötigt etwas Zeit
        yield env.timeout(benoetigter_kraftstoff/ AUFFUELLGESCHWINDIGKEIT)

        print(f'{env.now:6.1f} s: {name} wurde vollgetankt mit {benoetigter_kraftstoff:.1f} L')

Der Generator tankstellen_steuerung() wird definiert. In der Funktion wird der Tank der Tankstelle nachgefüllt

In [ ]:
def tankstellen_steuerung(env: simpy.Environment, tankstellen_tank: simpy.Container):
    """
    Überwacht regelmäßig den Füllstand des Tankstellentanks und ruft bei Bedarf den Tankwagen.

    Parameter:
        env (simpy.Environment): Die Simulationsumgebung, die den Zeitablauf steuert.
        tankstellen_tank (simpy.Container): Der Tank der Tankstelle, dessen Füllstand überwacht wird.

    Funktionsweise:
        - In einer Endlosschleife wird alle 10 Simulationssekunden der Füllstand überprüft.
        - Wenn der aktuelle Füllstand unter den definierten Schwellwert (SCHWELLENWERT) fällt,
          wird ein Tankwagen-Prozess gestartet, um den Tank nachzufüllen.
        - Während der Betankung pausiert die Steuerung, bis der Tankvorgang abgeschlossen ist.

    Zweck:
        Diese Funktion stellt sicher, dass der Tankstellentank nie vollständig leerläuft und
        automatisch bei niedrigem Füllstand nachgefüllt wird.
    """
    while True:
        if tankstellen_tank.level / tankstellen_tank.capacity * 100 < SCHWELLENWERT:
            # Der Tankwagen muss gerufen werden
            print(f'{env.now:6.1f} s: Ruf den Tankwagen')
            # Warten bis der Tankwagen ankommt und die Tankstelle befüllt
            yield env.process(tankwagen(env, tankstellen_tank))
        
        yield env.timeout(10) # Der Tank der Tankstelle wird alle 10 Sekunden überprüft

Der Generator tankwagen() wird definiert.

In [ ]:
def tankwagen(env: simpy.Environment, tankstellen_tank: simpy.Container):
    """Simuliert das Eintreffen und Befüllen der Tankstelle durch den Tankwagen.

    Parameter:
        env (simpy.Environment): Die Simulationsumgebung, die den Zeitablauf steuert.
        tankstellen_tank (simpy.Container): Der Tank der Tankstelle, der aufgefüllt werden soll.

    Funktionsweise:
        - Der Tankwagen benötigt eine feste Anfahrtszeit (TANKWAGEN_ZEIT), bevor er ankommt.
        - Nach Ablauf dieser Zeit wird die fehlende Tankmenge berechnet.
        - Anschließend wird die gesamte fehlende Menge in den Tankstellentank eingefüllt.
        - Es erfolgt eine Ausgabemeldung über den Zeitpunkt und die nachgefüllte Menge.

    Zweck:
        Diese Funktion modelliert den Nachschubprozess an der Tankstelle, also das Nachfüllen
        des Tanks durch den Tankwagen nach einer Anfahrtsverzögerung.
    """
    yield env.timeout(TANKWAGEN_ZEIT)
    nachfuellmenge = tankstellen_tank.capacity - tankstellen_tank.level     
    tankstellen_tank.put(nachfuellmenge)                                    # Die nachfuellmenge wird in den tankstellen_tank gepackt

    print(f'{env.now:6.1f} s: Tankwagen ist angekommen und hat den Tankstellentank mit {nachfuellmenge:.1f} L nachgefüllt')

Der Generator auto_generieren() wird definiert.

In [ ]:
def auto_generieren(env: simpy.Environment, tankstelle: simpy.Resource, tankstellen_tank: simpy.Container):
    """Erzeugt fortlaufend neue Autos, die an der Tankstelle ankommen und tanken möchten.

    Parameter:
        env (simpy.Environment): Die Simulationsumgebung, die den Zeitablauf steuert.
        tankstelle (simpy.Resource): Die Ressource, die die verfügbaren Zapfsäulen der Tankstelle repräsentiert.
        tankstellen_tank (simpy.Container): Der gemeinsame Tankvorrat, aus dem die Autos beim Tanken schöpfen.

    Funktionsweise:
        - In einer Endlosschleife wird nach einem zufälligen Zeitintervall (ANKUNFTSINTERVALL_AUTO)
          ein neues Auto erzeugt.
        - Jedes Auto wird als separater SimPy-Prozess gestartet und führt seine eigene Tanklogik aus.
        - Die Autos erhalten fortlaufende Nummern zur eindeutigen Identifikation.

    Zweck:
        Diese Funktion dient als Ereignisgenerator in der Simulation und sorgt dafür,
        dass während der gesamten Simulationszeit kontinuierlich neue Fahrzeuge an der Tankstelle eintreffen."""
    
    i = 0 
    while True:
        yield env.timeout(random.randint(*ANKUFNTSINTERVALL_AUTO))
        env.process(auto(f"Auto {i}", env, tankstelle, tankstellen_tank))
        i += 1

Einrichten und Start der Simulation

In [ ]:
print('Tankstelle nachfüllen')
random.seed(RANDOM_SEED)

Das Environment wird erstellt und die Prozesse werden gestartet

In [ ]:
env = simpy.Environment()
tankstelle = simpy.Resource(env, 2)
tankstellen_tank = simpy.Container(env, TANKSTELLEN_GROEßE, init = TANKSTELLEN_GROEßE)

env.process(tankstellen_steuerung(env, tankstellen_tank))
env.process(auto_generieren(env, tankstelle, tankstellen_tank))

Ausführen der Simulation

In [ ]:
env.run(until=SIMULATIONS_ZEIT)